In [1]:
# Step 0 --- check the workbench before we start.
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'mlflow']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'mlflow']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

Python  : 3.14.7
Folder  : MLOPS 
  numpy          ok
  pandas         ok
  sklearn        ok
  mlflow         ok

All good. You can carry on to Step 1.


---

## Step 0b --- the dataset

Every practical in this course uses the same 600 food deliveries.
The next cell makes sure the file is there.

In [2]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

dataset found   -> ../data/delivery_times.csv


---

# Walkthrough

Read each step, then run its cell.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)

orders = pd.read_csv(DATA)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X_train, X_test, y_train, y_test = train_test_split(
    orders[FEATURES], orders["delivery_min"],
    test_size=0.2, random_state=42,
)

for n_trees, depth in [(50, 4), (150, 8), (300, 12)]:
    m = RandomForestRegressor(n_estimators=n_trees, max_depth=depth,
                              random_state=42).fit(X_train, y_train)
    mae = mean_absolute_error(y_test, m.predict(X_test))
    print(f"n_estimators={n_trees:3d} max_depth={depth:2d} -> "
          f"MAE {mae:.3f}")

n_estimators= 50 max_depth= 4 -> MAE 3.529
n_estimators=150 max_depth= 8 -> MAE 2.427
n_estimators=300 max_depth=12 -> MAE 2.379


In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

WORK = Path("work")
WORK.mkdir(exist_ok=True)

orders = pd.read_csv(DATA)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X_train, X_test, y_train, y_test = train_test_split(
    orders[FEATURES], orders["delivery_min"],
    test_size=0.2, random_state=42,
)

for n_trees, depth in [(50, 4), (150, 8), (300, 12)]:
    m = RandomForestRegressor(n_estimators=n_trees, max_depth=depth,
                              random_state=42).fit(X_train, y_train)
    mae = mean_absolute_error(y_test, m.predict(X_test))
    print(f"n_estimators={n_trees:3d} max_depth={depth:2d} -> "
          f"MAE {mae:.3f}")

n_estimators= 50 max_depth= 4 -> MAE 3.529
n_estimators=150 max_depth= 8 -> MAE 2.427
n_estimators=300 max_depth=12 -> MAE 2.379


Those numbers exist only on your screen. Close the notebook and they
are gone. Now let us keep them.

In [5]:
import shutil

import mlflow
import mlflow.sklearn

# Teaching notebooks should start from a known state, so that your
# numbers match your instructor's. This clears any previous run of
# THIS notebook. You would never do this on a real project -- the
# whole value of the database is that it remembers.
db_path = (WORK / "mlflow.db").resolve()
db_path.unlink(missing_ok=True)
shutil.rmtree("mlartifacts", ignore_errors=True)

TRACKING_URI = f"sqlite:///{db_path.as_posix()}"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("delivery-time")

print("tracking store:", TRACKING_URI)

2026/10/01 00:38:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/01 00:38:17 INFO mlflow.store.db.utils: Updating database tables
2026/10/01 00:38:18 INFO mlflow.tracking.fluent: Experiment with name 'delivery-time' does not exist. Creating a new experiment.


tracking store: sqlite:////Users/muskan/Downloads/MLOPS /work/mlflow.db


In [6]:
with mlflow.start_run(run_name="first-try") as run:
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 50)
    mlflow.log_param("max_depth", 4)

    model = RandomForestRegressor(
        n_estimators=50, max_depth=4, random_state=42
    ).fit(X_train, y_train)

    mae = mean_absolute_error(y_test, model.predict(X_test))
    mlflow.log_metric("mae_minutes", mae)
    mlflow.set_tag("run_by", "student")

    first_run_id = run.info.run_id

print("run id :", first_run_id)
print("MAE    :", round(mae, 3))

run id : 1f7a3f0f414143d7924d812f341505f7
MAE    : 3.529


That run is now in the database permanently. Delete this notebook
and the record survives.

In [7]:
runs = mlflow.search_runs(experiment_names=["delivery-time"])

print("runs recorded:", len(runs))
print()
print(runs[["run_id", "params.n_estimators", "params.max_depth",
            "metrics.mae_minutes"]].to_string(index=False))

runs recorded: 1

                          run_id params.n_estimators params.max_depth  metrics.mae_minutes
1f7a3f0f414143d7924d812f341505f7                  50                4             3.528755


In [8]:
SETTINGS = [(50, 4), (150, 8), (300, 12), (100, 6), (200, None)]

for n_trees, depth in SETTINGS:
    with mlflow.start_run(run_name=f"rf-n{n_trees}-d{depth}"):
        mlflow.log_param("model_type", "RandomForestRegressor")
        mlflow.log_param("n_estimators", n_trees)
        mlflow.log_param("max_depth", depth)

        m = RandomForestRegressor(
            n_estimators=n_trees, max_depth=depth, random_state=42
        ).fit(X_train, y_train)

        preds = m.predict(X_test)
        mlflow.log_metric("mae_minutes",
                          mean_absolute_error(y_test, preds))
        mlflow.log_metric(
            "rmse_minutes",
            float(np.sqrt(((y_test - preds) ** 2).mean())),
        )
        print(f"  logged rf-n{n_trees}-d{depth}")

print("done")

  logged rf-n50-d4
  logged rf-n150-d8
  logged rf-n300-d12
  logged rf-n100-d6
  logged rf-n200-dNone
done


In [9]:
best_runs = mlflow.search_runs(
    experiment_names=["delivery-time"],
    order_by=["metrics.mae_minutes ASC"],
)

columns = ["tags.mlflow.runName", "params.n_estimators",
           "params.max_depth", "metrics.mae_minutes"]
print(best_runs[columns].head(6).to_string(index=False))

winner = best_runs.iloc[0]
print()
print("WINNER:", winner["tags.mlflow.runName"],
      "with MAE", round(winner["metrics.mae_minutes"], 3))

tags.mlflow.runName params.n_estimators params.max_depth  metrics.mae_minutes
        rf-n300-d12                 300               12             2.379200
      rf-n200-dNone                 200             None             2.412313
         rf-n150-d8                 150                8             2.426508
         rf-n100-d6                 100                6             2.614045
          rf-n50-d4                  50                4             3.528755
          first-try                  50                4             3.528755

WINNER: rf-n300-d12 with MAE 2.379


In [10]:
best_trees = int(winner["params.n_estimators"])
best_depth = winner["params.max_depth"]
best_depth = None if best_depth in ("None", None) else int(best_depth)

with mlflow.start_run(run_name="champion") as champion:
    mlflow.log_param("n_estimators", best_trees)
    mlflow.log_param("max_depth", best_depth)

    champion_model = RandomForestRegressor(
        n_estimators=best_trees,
        max_depth=best_depth,
        random_state=42,
    ).fit(X_train, y_train)

    mlflow.log_metric(
        "mae_minutes",
        mean_absolute_error(y_test, champion_model.predict(X_test)),
    )

    mlflow.sklearn.log_model(
        champion_model,
        name="model",
        input_example=X_train.head(3),
        registered_model_name="delivery-time-model",
        serialization_format="cloudpickle"
    )

    champion_id = champion.info.run_id

print("champion run:", champion_id)

2026/10/01 00:38:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/Users/muskan/Downloads/MLOPS /.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have miss

champion run: f41694779c0049d8bdcc664b8bdb56f3


Successfully registered model 'delivery-time-model'.
Created version '1' of model 'delivery-time-model'.


`registered_model_name` did two jobs at once: it stored the model
file, and it entered that file into the **registry** under a name.

In [11]:
from mlflow import MlflowClient

client = MlflowClient()

for mv in client.search_model_versions(
    "name='delivery-time-model'"
):
    print(f"name={mv.name}  version={mv.version}  "
          f"run_id={mv.run_id[:8]}...")

name=delivery-time-model  version=1  run_id=f4169477...


In [12]:
loaded = mlflow.sklearn.load_model("models:/delivery-time-model/1")

one_order = pd.DataFrame([{
    "distance_km": 5.0,
    "prep_time_min": 20,
    "traffic_level": 2,
    "rain": 0,
}])

print("loaded model type:", type(loaded).__name__)
print("prediction:", round(float(loaded.predict(one_order)[0]), 1),
      "minutes")

loaded model type: RandomForestRegressor
prediction: 42.9 minutes


In [13]:
print("Run this in a terminal, from the folder containing work/ :")
print()
print(f"  cd {Path.cwd()}")
print("  python -m mlflow ui --backend-store-uri "
      f"{TRACKING_URI} --port 5000")
print()
print("Then open http://127.0.0.1:5000")
print()
print("What you will see first: an empty 'Experiments' page.")
print("Click 'delivery-time' in the left sidebar to get the run")
print("table. Tick two runs and press Compare to see them side")
print("by side. The Models tab at the top shows the registry.")

Run this in a terminal, from the folder containing work/ :

  cd /Users/muskan/Downloads/MLOPS 
  python -m mlflow ui --backend-store-uri sqlite:////Users/muskan/Downloads/MLOPS /work/mlflow.db --port 5000

Then open http://127.0.0.1:5000

What you will see first: an empty 'Experiments' page.
Click 'delivery-time' in the left sidebar to get the run
table. Tick two runs and press Compare to see them side
by side. The Models tab at the top shows the registry.


Take a screenshot of the run table with the `mae_minutes` column
visible --- you need it for your submission.

### Task T1 --- Track a different kind of model


    Log one new run, named exactly `linear-baseline`, that:

    - trains a `LinearRegression` on `X_train` / `y_train`
    - logs a parameter `model_type` with the value `LinearRegression`
    - logs a metric `mae_minutes` with its error on the test set

    Store the run's id in `T1_run_id`.


> **Hint.** Copy Step 3 and change three things: the run name, the model, and the value of `model_type`. `LinearRegression` needs importing from `sklearn.linear_model`. Get the id with `run.info.run_id` inside the `with` block.

In [14]:
from sklearn.linear_model import LinearRegression

T1_run_id = None

with mlflow.start_run(run_name="linear-baseline") as run:
    model = LinearRegression()
    model.fit(X_train, y_train)

    mlflow.log_param("model_type", "LinearRegression")

    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)

    mlflow.log_metric("mae_minutes", mae)

    T1_run_id = run.info.run_id

print("T1_run_id =", T1_run_id)

T1_run_id = 53158f4be2474978b6eb885b2c934d1f


### Task T2 --- Ask the database, not your memory


    Without reading any earlier output, use `mlflow.search_runs` to find the
    run with the **lowest** `mae_minutes` across the whole `delivery-time`
    experiment.

    Put its run name in `T2_best_name` and its error in `T2_best_mae`.


> **Hint.** Step 6 shows the pattern: pass `order_by=["metrics.mae_minutes ASC"]`, then take `.iloc[0]`. The run name lives in the column `tags.mlflow.runName`.

In [15]:
T2_runs = mlflow.search_runs(
    experiment_ids=[
        mlflow.get_experiment_by_name("delivery-time").experiment_id
    ],
    order_by=["metrics.mae_minutes ASC"]
)

T2_best_name = T2_runs.iloc[0]["tags.mlflow.runName"]
T2_best_mae = T2_runs.iloc[0]["metrics.mae_minutes"]

print(T2_best_name, T2_best_mae)

linear-baseline 1.924653079833835


### Task T3 --- A sweep you can filter later


    Run a small sweep of `DecisionTreeRegressor` at depths `[2, 4, 6]`.
    For each depth, log one run that:

    - is tagged `family` = `tree`
    - logs the parameter `max_depth`
    - logs the metric `mae_minutes`

    Then query MLflow for runs with that tag and put the count in
    `T3_tree_runs`.


> **Hint.** `mlflow.set_tag("family", "tree")` goes inside the `with` block. To find them again, use `filter_string="tags.family = 'tree'"` in `search_runs`. The count is just `len(...)` of the result.

In [16]:
from sklearn.tree import DecisionTreeRegressor

for depth in [2, 4, 6]:
    with mlflow.start_run(run_name=f"tree-depth-{depth}"):
        mlflow.set_tag("family", "tree")

        mlflow.log_param("max_depth", depth)

        model = DecisionTreeRegressor(
            max_depth=depth,
            random_state=42
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        mae = mean_absolute_error(y_test, predictions)

        mlflow.log_metric("mae_minutes", mae)

T3_tree_runs = len(
    mlflow.search_runs(
        experiment_ids=[
            mlflow.get_experiment_by_name("delivery-time").experiment_id
        ],
        filter_string="tags.family = 'tree'"
    )
)

print("runs tagged family=tree:", T3_tree_runs)

runs tagged family=tree: 3


---

## Self-check

Run the cell below to mark your work.

In [17]:
# ------------------------------------------------------------------
# SELF-CHECK --- run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | a run named linear-baseline exists in MLflow', lambda: len(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.mlflow.runName = 'linear-baseline'")) >= 1)
_check("T1 | T1_run_id is that run's id", lambda: T1_run_id in set(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.mlflow.runName = 'linear-baseline'")['run_id']))
_check('T1 | it logged model_type and a sensible mae_minutes', lambda: (lambda r: r['params.model_type'].iloc[0] == 'LinearRegression' and 0 < float(r['metrics.mae_minutes'].iloc[0]) < 15)(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.mlflow.runName = 'linear-baseline'")))
_check('T2 | T2_best_mae really is the smallest error recorded', lambda: abs(float(T2_best_mae) - float(mlflow.search_runs(experiment_names=['delivery-time'])['metrics.mae_minutes'].min())) < 1e-6)
_check('T2 | T2_best_name is the run that holds it', lambda: T2_best_name == mlflow.search_runs(experiment_names=['delivery-time'], order_by=['metrics.mae_minutes ASC']).iloc[0]['tags.mlflow.runName'])
_check('T3 | three runs are tagged family=tree', lambda: len(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.family = 'tree'")) == 3)
_check('T3 | T3_tree_runs reports that count', lambda: int(T3_tree_runs) == 3)
_check('T3 | each tagged run logged a max_depth of 2, 4 or 6', lambda: set(mlflow.search_runs(experiment_names=['delivery-time'], filter_string="tags.family = 'tree'")['params.max_depth']) == {'2', '4', '6'})

print("==================================================================")
print("SELF-CHECK   Practical 06 --- Remembering Every Experiment")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

SELF-CHECK   Practical 06 --- Remembering Every Experiment
  [PASS]  T1 | a run named linear-baseline exists in MLflow
  [PASS]  T1 | T1_run_id is that run's id
  [PASS]  T1 | it logged model_type and a sensible mae_minutes
  [PASS]  T2 | T2_best_mae really is the smallest error recorded
  [PASS]  T2 | T2_best_name is the run that holds it
  [PASS]  T3 | three runs are tagged family=tree
  [PASS]  T3 | T3_tree_runs reports that count
  [PASS]  T3 | each tagged run logged a max_depth of 2, 4 or 6
------------------------------------------------------------------
  8 of 8 checks passed
Well done. Save the notebook and submit it.


In [18]:
all_runs = mlflow.search_runs(
    experiment_names=["delivery-time"],
    order_by=["metrics.mae_minutes ASC"],
).dropna(subset=["metrics.mae_minutes"])

best = all_runs.iloc[0]
worst = all_runs.iloc[-1]
diff = worst["metrics.mae_minutes"] - best["metrics.mae_minutes"]

print("BEST :", best["tags.mlflow.runName"], round(best["metrics.mae_minutes"], 3))
print("WORST:", worst["tags.mlflow.runName"], round(worst["metrics.mae_minutes"], 3))
print("Difference:", round(diff, 3), "minutes")

BEST : linear-baseline 1.925
WORST: tree-depth-2 5.834
Difference: 3.909 minutes


The best run was **linear-baseline (LinearRegression)** with an MAE of **1.925 minutes**, which beat the worst run, **tree-depth-2 (DecisionTreeRegressor, max_depth=2)** with an MAE of **5.834 minutes**, by **3.909 minutes**.